# Core API's

In [1]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

import evaluate

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

## AutoTokenizer

In [9]:
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
print(tokenizer)
print(tokenizer.vocab_size)
print(tokenizer.model_max_length)

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})
30522
512


In [10]:
text = "I reallyyy enjoyed this movie."

tokens = tokenizer(text)
print(tokens)

{'input_ids': [101, 1045, 2428, 2100, 2100, 5632, 2023, 3185, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [11]:
print("Input IDs:")
print(tokens["input_ids"])

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"]))


Input IDs:
[101, 1045, 2428, 2100, 2100, 5632, 2023, 3185, 1012, 102]

Tokens:
['[CLS]', 'i', 'really', '##y', '##y', 'enjoyed', 'this', 'movie', '.', '[SEP]']


## Auto Model

In [13]:
texts = [
    "I loved this movie so much.",
    "This movie was terrible.",
    "The acting was excellent."
]

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)

Input IDs shape: torch.Size([3, 9])
Attention mask shape: torch.Size([3, 9])


In [14]:
base_model = AutoModel.from_pretrained(model_name)

base_model.to(device)
base_model.eval()

print(type(base_model))


Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yuvan\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 908.35it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert

<class 'transformers.models.distilbert.modeling_distilbert.DistilBertModel'>


In [17]:
inputs = {
    key: value.to(device)
    for key,value in batch.items()
}

with torch.no_grad():
    outputs = base_model(**inputs)

print("Last Hidden State Shape: ")
print(outputs.last_hidden_state.shape)

# Here 3 is Batch size
# 9 is sequence len after padding
# 768 embedding size

Last Hidden State Shape: 
torch.Size([3, 9, 768])


'\n  Here 3 is Batch size\n  9 is sequence len after padding\n  768 embedding size\n'

## AutoSequence Classification

In [19]:
sequence_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels= 2
)

print(type(sequence_model))

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 928.85it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'>


In [20]:
sequence_model.to(device)
sequence_model.eval()

with torch.no_grad():
    outputs = sequence_model(**inputs)

print("Logits shape:", outputs.logits.shape)
print("Logits:")
print(outputs.logits)

Logits shape: torch.Size([3, 2])
Logits:
tensor([[-0.0508,  0.0319],
        [-0.0384,  0.0339],
        [-0.0559,  0.0443]], device='cuda:0')


In [21]:
probabilities = torch.softmax(outputs.logits, dim=-1)

print("Probabilities:")
print(probabilities)

print("\nPredicted class:")
print(torch.argmax(probabilities, dim=-1))


Probabilities:
tensor([[0.4793, 0.5207],
        [0.4819, 0.5181],
        [0.4750, 0.5250]], device='cuda:0')

Predicted class:
tensor([1, 1, 1], device='cuda:0')
